In [5]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_pareto

from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD
from scipy.stats import kruskal

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import statistics
import scikit_posthocs as sp

# Hypervolume:

In [83]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_names = ['dtlz1']
objective_dim = 3
position_dim = 10

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2, 3]
choose_Xr_pool_type = [0,1,2] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Consider other results
insert_nsga2 = True
insert_old_mesh = True
######################################################

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)

In [84]:
# List of file tuples
file_tuple_mesh = []
file_tuple_old_mesh = []
file_tuple_nsga2 = []

# Name of chosen files
file_tuple_mesh = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}', exp_name)
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for exp_name in experiment_names
    ]

# Name of other files
if insert_old_mesh:
	file_tuple_old_mesh = [(f"OLD_MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1} (Original)', exp_name)
						for i in choose_global_best_attribution_type
						for j in choose_Xr_pool_type
						for k in choose_DE_mutation_type
						for exp_name in experiment_names
                        if os.path.exists(f"../results/OLD_MESH_G{i+1}S{j+1}D{k+1}_{exp_name}_{objective_dim}_{position_dim}.pkl")
					]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]

file_tuple = file_tuple_mesh + file_tuple_old_mesh + file_tuple_nsga2

# Take the results
results_mesh_hv = []
results_old_mesh_hv = []
results_nsga2_hv = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_old_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_old_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open("../results/" + file_name, "rb") as f:
		results_nsga2_hv.append(pickle.load(f))
results = results_mesh_hv + results_old_mesh_hv + results_nsga2_hv

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Median', 'Combined HV', 'Min. HV', 'Max. HV']

# Store the datas
data = [[
		alg_name,
		func_name.upper(),
		statistics.mean(HVs := [indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)]),
		statistics.stdev(HVs) if len(HVs) > 1 else 0,
		statistics.median(HVs) if len(HVs) > 1 else HVs[0],
		indicator(results[i]['combined'][1]),
		min(HVs),
		max(HVs),
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
		for r in [results[i]]
	]

# Create the DataFrame with hypervolume results
df = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df.sort_values(by=['Median'], ascending=False))

,Algorithm,Function,Mean HV,Std. Dev.,Median,Combined HV,Min. HV,Max. HV
20,MESH - G2S2D1,DTLZ1,995.492150,5.545243,997.459465,999.940389,972.453501,999.940389
10,MESH - G1S3D1,DTLZ1,991.541802,19.488570,997.438431,999.956842,896.698014,999.957815
15,MESH - G2S1D1,DTLZ1,988.607110,39.344440,997.029950,999.960084,782.098526,999.960084
5,MESH - G1S2D1,DTLZ1,995.902682,4.098500,996.874570,999.963153,984.112723,999.963371
22,MESH - G2S2D3,DTLZ1,978.912028,29.376799,993.177037,999.754039,883.924234,999.754039
12,MESH - G1S3D3,DTLZ1,978.330921,33.742471,991.988415,999.806051,839.341580,999.740966
1,MESH - G1S1D2,DTLZ1,989.231371,11.042631,991.720353,999.879791,959.623238,999.879791
0,MESH - G1S1D1,DTLZ1,986.846438,19.038454,991.345586,999.810966,899.504893,999.809604
11,MESH - G1S3D2,DTLZ1,973.589612,48.396220,990.015264,999.398725,742.856357,999.398725
2,MESH - G1S1D3,DTLZ1,952.697664,85.760329,986.612466,999.579984,589.744001,999.580410


In [ ]:
print(df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

## Hypervolume Boxplot:

In [ ]:
# Experiment name
experiment_name = 'zdt4'
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]
# Consider NSGA-II results
insert_nsga2 = False


# Name of chosen files
file_names_mesh = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
    ]
# Name of NSGA-II files
file_names_nsga2 = []
if insert_nsga2:
  file_names_nsga2.extend(['NSGA2_' + experiment_name + '.pkl'])

# Take the results
results_mesh = []
results_nsga2 = []
for i in range(len(file_names_mesh)):
  with open("../results/" + file_names_mesh[i], "rb") as f:
    results_mesh.append(pickle.load(f))
for i in range(len(file_names_nsga2)):
  with open("../results/" + file_names_nsga2[i], "rb") as f:
    results_nsga2.append(pickle.load(f))

# Get MESH hypervolumes
data = []
for i, fn in enumerate(file_names_mesh):
  HVs = []
  r = results_mesh[i]
  for j in range(len(results_mesh[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)
# Get NSGA-II hypervolumes if applicable
for i in range(len(file_names_nsga2)):
  HVs = []
  r = results_nsga2[i]
  for j in range(len(results_nsga2[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)

# Creating a boxplot
labels = ['MESH - ' + fn.split('_', 1)[0] for fn in file_names_mesh] + (['NSGA-II'] if insert_nsga2 else [])
plt.figure(figsize=(8, 5))
plt.boxplot(data, tick_labels=labels, showmeans=False)

# Titles and labels
# plt.title('Hypervolume - ' + experiment_name.upper(), fontsize=14)
plt.xlabel('Algoritmos') # plt.xlabel('Algorithms')
plt.ylabel('Hipervolume') # plt.ylabel('Hypervolume')
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.show()

# Inverse Generational Distance:

In [ ]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_name = 'zdt1'
objective_dim = 2
position_dim = 10
wfg_k = 6
n_pareto_density = 100

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2, 3]
choose_Xr_pool_type = [0,1,2] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Consider other results
insert_nsga2 = True
insert_old_mesh = True
######################################################

# Create the hypervolume indicator
indicator = IGD(get_pareto(experiment_name, n_pareto_density, position_dim, objective_dim, wfg_k))

In [ ]:
# List of file tuples
file_tuple_mesh = []
file_tuple_old_mesh = []
file_tuple_nsga2 = []

# Name of chosen files
file_tuple_mesh = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}', experiment_name)
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
    ]

# Name of other files
if insert_old_mesh:
	file_tuple_old_mesh = [(f"OLD_MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1} (Original)', experiment_name)
						for i in choose_global_best_attribution_type
						for j in choose_Xr_pool_type
						for k in choose_DE_mutation_type
                        if os.path.exists(f"../results/OLD_MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl")
					]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', experiment_name)]

file_tuple = file_tuple_mesh + file_tuple_old_mesh + file_tuple_nsga2

# Take the results
results_mesh_igd = []
results_old_mesh_igd = []
results_nsga2_igd = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_mesh_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_old_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_old_mesh_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open("../results/" + file_name, "rb") as f:
		results_nsga2_igd.append(pickle.load(f))
results = results_mesh_igd + results_old_mesh_igd + results_nsga2_igd

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean IGD', 'Std. Dev.', 'Median', 'Combined IGD', 'Min. IGD', 'Max. IGD']

# Store the datas
data = [[
		alg_name,
		func_name.upper(),
		statistics.mean(IGDs := [indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)]),
		statistics.stdev(IGDs) if len(IGDs) > 1 else 0,
		statistics.median(IGDs) if len(IGDs) > 1 else IGDs[0],
		indicator(results[i]['combined'][1]),
		min(IGDs),
		max(IGDs),
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
		for r in [results[i]]
	]

# Create the DataFrame with hypervolume results
df = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df.sort_values(by=['Median'], ascending=True))

,Algorithm,Function,Mean IGD,Std. Dev.,Median,Combined IGD,Min. IGD,Max. IGD
24,MESH - G2S2D5,ZDT1,0.004348,0.000230,0.004328,0.007780,0.003973,0.004723
15,MESH - G2S1D1,ZDT1,0.004321,0.000276,0.004340,0.008496,0.003692,0.004792
27,MESH - G2S3D3,ZDT1,0.004373,0.000181,0.004367,0.007660,0.004009,0.004653
8,MESH - G1S2D4,ZDT1,0.004406,0.000244,0.004372,0.007355,0.003940,0.004925
20,MESH - G2S2D1,ZDT1,0.004412,0.000282,0.004378,0.007567,0.003878,0.004943
19,MESH - G2S1D5,ZDT1,0.004453,0.000235,0.004380,0.007328,0.004087,0.005010
7,MESH - G1S2D3,ZDT1,0.004416,0.000226,0.004389,0.006872,0.004078,0.004889
10,MESH - G1S3D1,ZDT1,0.004415,0.000293,0.004389,0.008774,0.003782,0.004992
2,MESH - G1S1D3,ZDT1,0.004446,0.000290,0.004405,0.007760,0.003876,0.005191
14,MESH - G1S3D5,ZDT1,0.004396,0.000251,0.004416,0.007086,0.003936,0.004824


# Hypothesis Test

## Auxiliar Function

In [ ]:
def rank_configurations(data, alpha=0.05, ascending=True):
    """
    Rank algorithm configurations using Kruskal-Wallis + Dunn post-hoc test with Holm p-value correction.
    Configurations that are not significantly different are assigned the same rank.

    Parameters
    ----------
    data : list of lists
        Each element is a list with the hypervolume values of one configuration.
    alpha : float
        Significance level for hypothesis tests.
    ascending : bool
        If True, higher medians are ranked better (rank 1 is the best).
        If False, lower medians are ranked better.

    Returns
    -------
    results : dict
        'kruskal_pval' : float
            p-value from Kruskal-Wallis test.
        'posthoc_pvalues' : DataFrame or None
            Adjusted p-values from Dunn test (only if Kruskal-Wallis is significant).
        'ranking' : list of int
            Rank of each configuration (1 = best), ties receive the same rank.
    """

    n_configs = len(data)
    rankings = [1] * n_configs
    posthoc_pvalues = None

    # Kruskal-Wallis test
    stat, kruskal_pval = kruskal(*data)
    if kruskal_pval < alpha:
        # Build DataFrame for Dunn test
        df = pd.DataFrame({
            "value": np.concatenate(data),
            "group": np.concatenate([[i]*len(d) for i, d in enumerate(data)])
        })
        # Dunn's post-hoc with Holm correction
        posthoc_pvalues = sp.posthoc_dunn(df, val_col="value", group_col="group", p_adjust="holm")
        # Sort configurations by median
        medians = [(i, np.median(d)) for i, d in enumerate(data)]
        medians.sort(key=lambda x: x[1], reverse=ascending)
        # Build ranking with ties
        rankings = [None] * n_configs
        current_rank = 1
        current_group = [medians[0][0]]
        # Calculating the rankings
        for idx in range(1, n_configs):
            curr_idx = medians[idx][0]
            # Check if current index ties with all members of current_group
            is_tie = all(posthoc_pvalues.loc[prev_idx, curr_idx] > alpha for prev_idx in current_group)
            if is_tie:
                current_group.append(curr_idx)
            else:
                for g in current_group:
                    rankings[g] = current_rank
                current_rank += 1
                current_group = [curr_idx]
        # Assign rank to last group
        for g in current_group:
            rankings[g] = current_rank
    # Return the dictionary with the kruskal p-value and the rankings
    return {
        "kruskal_pvalue": kruskal_pval,
        "ranking": rankings
    }

## Test for Hypervolume

In [81]:
#################### CUSTOMIZABLE ####################
# Significance level
alpha_hv = 0.05

# Experiment configuration
experiment_name = 'dtlz1'
objective_dim = 3
position_dim = 10

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2, 3]
choose_Xr_pool_type = [0,1,2] # [0, 1, 2]
choose_DE_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Consider other results
insert_nsga2 = True
insert_old_mesh = True
######################################################

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)

In [86]:
# List of file tuples
file_tuple_mesh = []
file_tuple_old_mesh = []
file_tuple_nsga2 = []

# Name of chosen files
file_tuple_mesh = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}', experiment_name)
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
    ]

# Name of other files
if insert_old_mesh:
	file_tuple_old_mesh = [(f"OLD_MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1} (Original)', experiment_name)
						for i in choose_global_best_attribution_type
						for j in choose_Xr_pool_type
						for k in choose_DE_mutation_type
                        if os.path.exists(f"../results/OLD_MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl")
					]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', experiment_name)]

file_tuple = file_tuple_mesh + file_tuple_old_mesh + file_tuple_nsga2

# Take the results
results_mesh_hv = []
results_old_mesh_hv = []
results_nsga2_hv = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_old_mesh:
	with open("../results/" + file_name, "rb") as f:
		results_old_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open("../results/" + file_name, "rb") as f:
		results_nsga2_hv.append(pickle.load(f))
results = results_mesh_hv + results_old_mesh_hv + results_nsga2_hv

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Kruskal-Wallis p-value', 'Ranking']


hv_values = [[indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)] for i in range(len(results))]

# Store the datas
test_hv_dict = rank_configurations(hv_values, alpha_hv)
data = [[
		alg_name,
		func_name.upper(),
		test_hv_dict['kruskal_pvalue'],
		# test_hv_dict['posthoc_pvalues'][i],
		test_hv_dict['ranking'][i]
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
	]

df = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df.sort_values(by=['Ranking'], ascending=True))

,Algorithm,Function,Kruskal-Wallis p-value,Ranking
0,MESH - G1S1D1,DTLZ1,3.987394e-288,1
1,MESH - G1S1D2,DTLZ1,3.987394e-288,1
2,MESH - G1S1D3,DTLZ1,3.987394e-288,1
4,MESH - G1S1D5,DTLZ1,3.987394e-288,1
6,MESH - G1S2D2,DTLZ1,3.987394e-288,1
5,MESH - G1S2D1,DTLZ1,3.987394e-288,1
7,MESH - G1S2D3,DTLZ1,3.987394e-288,1
11,MESH - G1S3D2,DTLZ1,3.987394e-288,1
15,MESH - G2S1D1,DTLZ1,3.987394e-288,1
10,MESH - G1S3D1,DTLZ1,3.987394e-288,1
